# Data agent evaluation

Runs the question bank against the published Contoso Coffee data agent,
grades every answer against ground truth, repeats each question so that
nondeterminism is visible, and writes the results to Delta tables that
Activator watches.

This notebook is **generated**. Edit `validation/eval_harness.py` or
`validation/agent_client.py` and run `python validation/build_eval_notebook.py`.
Editing the notebook directly will be overwritten.

## What it does not do

It never changes the model or the agent. It proposes fixes and records
them. A human decides. Two rules make that non negotiable:

1. Agent instructions are not passed to the DAX generation step for a
   semantic model source, so editing the agent cannot fix a wrong number.
2. A loop allowed to write verified answers would pin its way to a
   perfect score over a model that is still wrong.


## 1. Parameters

No `%pip install` anywhere in this notebook, which is deliberate. The
agent is reached over plain JSON-RPC with the standard library. Adding
the `mcp` package to a Spark session upgrades pydantic, anyio,
typing-extensions and jsonschema over the builds the runtime ships,
and a scheduled job cannot afford that kind of instability.

This cell is tagged as the parameters cell, so a pipeline or a
scheduled run can override any of it.

In [ ]:
WORKSPACE_ID = "1713f459-7fcf-4704-94d6-7df5827ddcb0"
DATA_AGENT_ID = "f025126c-ae31-4e51-86c4-a1bcb6949061"
LAKEHOUSE_NAME = "LH_ContosoCoffee"

# Repetitions per question. This is the single most valuable knob in
# the notebook. At 1 you cannot tell a model that is wrong from a
# model that is ambiguous, and the second is worse in front of an
# audience because you cannot predict it or brief around it.
REPEAT = 3

# Questions in flight at once. The agent is a shared capacity
# resource and throttling looks exactly like a flake, which would
# poison the one signal this notebook exists to produce.
CONCURRENCY = 3

TABLE_PREFIX = "eval"
SURFACE = "D"  # pass D in validation/scorecard.md, the data agent

# Eventhouse that Activator watches. Delta holds the history and is
# what you query. Kusto is the event spine that makes alerting
# possible, because Activator cannot watch a Delta table directly.
KUSTO_URI = "https://trd-391auppsxutg30p2va.z9.kusto.fabric.microsoft.com"
KUSTO_DB = "EH_AgentEval"


## 2. Embedded harness

Generated from `validation/eval_harness.py`. Pure logic, no Fabric
imports, covered by unit tests that run on a laptop with no capacity.

In [ ]:
import re
import statistics
from dataclasses import dataclass, field

# --------------------------------------------------------------------------
# Grades
# --------------------------------------------------------------------------

CORRECT = "Correct"
PARTLY_CORRECT = "Partly correct"
WRONG = "Wrong"
REFUSED = "Refused"
ERRORED = "Errored"

STABLE_PASS = "stable_pass"
STABLE_FAILURE = "stable_failure"
FLAKE = "flake"
ERRORED_RUN = "errored"

SCORED = "scored"
PROBE = "probe"

# --------------------------------------------------------------------------
# Tolerances
# --------------------------------------------------------------------------
#
# These are the whole argument of the demo expressed as numbers, so they are
# worth stating plainly.
#
# The failure this demo exists to catch is a model answering with Gross Sales
# instead of Total Net Sales. On this dataset that is an error of roughly one
# to three percent. So the tolerance has to be tight enough to call that
# Wrong, and loose enough to accept an agent that rounds a large total to the
# nearest dollar. MONEY_REL_TOLERANCE of 0.0005 is 0.05 percent, which is two
# orders of magnitude below the error we are hunting.

MONEY_ABS_TOLERANCE = 0.51  # accepts rounding to the nearest dollar
MONEY_REL_TOLERANCE = 0.0005  # 0.05 percent
PERCENT_TOLERANCE = 0.06  # percentage points, accepts one decimal rounding
COUNT_TOLERANCE = 0  # units are integers, so be exact

REFUSAL_PATTERNS = [
    r"\bi (?:can|could)(?:no|n[o']?)t\b",
    r"\bcannot\b",
    r"\bcan't\b",
    r"\bunable to\b",
    r"\bdo(?:es)? not (?:contain|include|have)\b",
    r"\bdon'?t have\b",
    r"\bno data\b",
    r"\bnot available\b",
    r"\bhistorical data only\b",
    r"\bonly (?:contains|includes|has) historical\b",
    r"\bthere is no\b",
    r"\bdoes not exist\b",
    r"\bnot present in\b",
    r"\bnot a valid\b",
    r"\bis ?n[o']?t (?:a )?valid\b",
    r"\bno such\b",
    r"\bnot one of\b",
    r"\binvalid\b",
]

CLARIFY_PATTERNS = [
    r"\bdid you mean\b",
    r"\bdo you mean\b",
    r"\bcould you clarify\b",
    r"\bwhich (?:one|of these|do you)\b",
    r"\bambiguous\b",
    r"\bcan refer to\b",
    r"\btwo ways\b",
    r"\bassum(?:e|ed|ing)\b",
    r"\binterpret(?:ed)? (?:this|that|it) as\b",
    r"\bi used\b",
]

# Per-probe acceptance rules, written straight from the "good outcome" column
# of validation/question-bank.md. Generic refusal detection is not precise
# enough here: a probe can be answered well in prose that contains no refusal
# language at all, and a grader that cries wolf on a correct answer destroys
# trust in the alerting faster than a grader that misses one.
PROBE_ACCEPT = {
    # "Says the model contains historical data only, does not project"
    "F01": [
        r"\bhistorical\b",
        r"\bcannot (?:forecast|project|predict|provide a forecast)\b",
        r"\bno (?:forecast|projection|predictive)\b",
        r"\bdoes not (?:support|include) (?:forecast|projection)",
        *REFUSAL_PATTERNS,
    ],
    # "Asks whether you mean margin dollars or margin rate, or states that it
    # used margin dollars". A currency figure attached to a named margin
    # measure is a statement that it used margin dollars, so it counts. Naming
    # no measure at all does not.
    "F02": [
        r"margin.{0,160}(?:dollars?|rate|percent)",
        r"(?:dollars?|rate|percent).{0,160}margin",
        r"margin[^.]{0,120}[$£€]",
        r"[$£€][^.]{0,120}margin",
        *CLARIFY_PATTERNS,
    ],
    # "Says there is no Northwest region, and lists the three that exist"
    "F03": [
        r"\bnot a valid\b",
        r"\bis ?n[o']?t (?:a )?valid\b",
        r"\bno such\b",
        r"\bthere is no\b",
        r"\bdoes not exist\b",
        r"\bnot one of\b",
        r"\binvalid\b",
        r"\bnot (?:a )?region\b",
    ],
}

# Language that shows the model narrowed the time scope without being asked.
# Every one of these was observed in the wild on questions that carried no
# time filter at all.
TIME_NARROWING_PATTERNS = [
    r"\bmost recent\b",
    r"\blatest\b",
    r"\bcurrent (?:month|year|period)\b",
    r"\bfor the (?:last|past) (?:month|year|quarter)\b",
    r"\bin that period\b",
]

# The agent failing is not the model being wrong. Conflating the two puts
# infrastructure noise into a metric that is supposed to measure modelling,
# and a metric people learn to discount is worse than no metric.
AGENT_FAILURE_PATTERNS = [
    r"\bdata agent run failed\b",
    r"\bfailed before producing\b",
    r"\ban error occurred while\b",
    r"\binternal server error\b",
    r"\brequest (?:timed out|failed)\b",
    r"\bservice unavailable\b",
    r"\btry again later\b",
]

MONTH_NAMES = {
    "01": "January", "02": "February", "03": "March", "04": "April",
    "05": "May", "06": "June", "07": "July", "08": "August",
    "09": "September", "10": "October", "11": "November", "12": "December",
}


# --------------------------------------------------------------------------
# Question bank
# --------------------------------------------------------------------------

@dataclass(frozen=True)
class Question:
    id: str
    text: str
    tests: str
    kind: str  # SCORED or PROBE


_ROW = re.compile(r"^\|\s*(Q\d{2}|F\d{2})\s*\|\s*(.+?)\s*\|\s*(.+?)\s*\|\s*$")


def parse_question_bank(markdown: str) -> list[Question]:
    """Read the questions out of validation/question-bank.md.

    Parsing the markdown rather than duplicating the questions in code is the
    point. A question asked by the harness and a question printed in the docs
    that drift apart is a silent, and very confusing, failure.
    """
    questions: list[Question] = []
    seen: set[str] = set()

    for line in markdown.splitlines():
        match = _ROW.match(line.strip())
        if not match:
            continue
        qid, text, tests = match.group(1), match.group(2), match.group(3)
        if qid in seen:
            continue
        seen.add(qid)
        questions.append(
            Question(
                id=qid,
                text=text.strip(),
                tests=tests.strip(),
                kind=SCORED if qid.startswith("Q") else PROBE,
            )
        )

    return sorted(questions, key=lambda q: (q.kind != SCORED, q.id))


# --------------------------------------------------------------------------
# Expectations
# --------------------------------------------------------------------------

@dataclass(frozen=True)
class Expected:
    """One machine-checkable expectation.

    values: numbers that must all appear in the answer, as (number, kind).
    labels: groups of alternative strings. Every group must match at least one
            of its alternatives, which is how "June" and "2025-06" can both be
            accepted for the same answer.
    probe_kind: for F01 to F03, what good behaviour looks like.
    """

    id: str
    values: tuple[tuple[float, str], ...] = ()
    labels: tuple[tuple[str, ...], ...] = ()
    forbidden: tuple[str, ...] = ()
    probe_kind: str | None = None
    probe_accept: tuple[str, ...] = ()


def build_expectations(raw: dict) -> dict[str, Expected]:
    """Turn ground_truth.compute_raw() into expectations, per question."""
    top_store_name, top_store_value = raw["top_store"]
    top_product_name, top_product_value = raw["top_product"]
    best_month_key, best_month_value = raw["best_month_2025"]

    month_label = MONTH_NAMES.get(best_month_key.split("-")[1], best_month_key)

    def money_group(mapping: dict[str, float]) -> tuple:
        return tuple((value, "money") for value in mapping.values())

    def label_group(mapping: dict[str, float]) -> tuple:
        return tuple((key,) for key in mapping)

    expectations = {
        "Q01": Expected("Q01", ((raw["total_net"], "money"),)),
        "Q02": Expected("Q02", ((raw["total_margin"], "money"),)),
        "Q03": Expected("Q03", ((raw["margin_pct"] * 100, "percent"),)),
        "Q04": Expected("Q04", ((raw["total_units"], "count"),)),
        "Q05": Expected("Q05", ((raw["net_2024"], "money"),)),
        "Q06": Expected("Q06", ((raw["net_2025"], "money"),)),
        "Q07": Expected("Q07", ((raw["yoy_pct"] * 100, "percent"),)),
        "Q08": Expected(
            "Q08", ((top_store_value, "money"),), ((top_store_name,),)
        ),
        "Q09": Expected(
            "Q09", ((top_product_value, "money"),), ((top_product_name,),)
        ),
        "Q10": Expected(
            "Q10", money_group(raw["by_region"]), label_group(raw["by_region"])
        ),
        "Q11": Expected(
            "Q11", money_group(raw["by_category"]), label_group(raw["by_category"])
        ),
        "Q12": Expected(
            "Q12", money_group(raw["by_channel"]), label_group(raw["by_channel"])
        ),
        "Q13": Expected(
            "Q13",
            ((best_month_value, "money"),),
            ((best_month_key, month_label),),
        ),
        "Q14": Expected(
            "Q14",
            ((raw["weekend_net"], "money"), (raw["weekday_net"], "money")),
            (("weekend",), ("weekday",)),
        ),
        "Q15": Expected("Q15", ((raw["avg_order_line"], "money"),)),
        # The probes. A value here is a failure, not a success.
        "F01": Expected(
            "F01", probe_kind="refuse", probe_accept=tuple(PROBE_ACCEPT["F01"])
        ),
        "F02": Expected(
            "F02", probe_kind="clarify", probe_accept=tuple(PROBE_ACCEPT["F02"])
        ),
        "F03": Expected(
            "F03",
            probe_kind="refuse",
            forbidden=("northwest",),
            probe_accept=tuple(PROBE_ACCEPT["F03"]),
        ),
    }
    return expectations


# --------------------------------------------------------------------------
# Number extraction
# --------------------------------------------------------------------------

_NUMBER = re.compile(
    r"(?P<currency>[$£€])?\s*"
    r"(?P<number>\d{1,3}(?:,\d{3})+(?:\.\d+)?|\d+(?:\.\d+)?)"
    r"\s*(?P<suffix>%|percent|percentage points?|pp|[KMB]\b)?",
    re.IGNORECASE,
)


def extract_numbers(text: str) -> list[tuple[float, str]]:
    """Pull every number out of free text, tagged as money, percent or bare.

    A number can be reported more than once with different tags. "$1.2M" is
    money 1200000. "5%" is percent 5. A bare "94,417" is tagged bare so that
    it can satisfy a count or, if nothing better matches, a money expectation.
    """
    found: list[tuple[float, str]] = []

    for match in _NUMBER.finditer(text or ""):
        raw = match.group("number").replace(",", "")
        try:
            value = float(raw)
        except ValueError:
            continue

        currency = match.group("currency")
        suffix = (match.group("suffix") or "").lower()

        multiplier = 1.0
        if suffix == "k":
            multiplier = 1_000.0
        elif suffix == "m":
            multiplier = 1_000_000.0
        elif suffix == "b":
            multiplier = 1_000_000_000.0

        if suffix in {"%", "percent", "percentage point", "percentage points", "pp"}:
            found.append((value, "percent"))
        elif currency:
            found.append((value * multiplier, "money"))
        elif multiplier != 1.0:
            found.append((value * multiplier, "bare"))
        else:
            found.append((value, "bare"))

    return found


def matches_value(expected: float, kind: str, candidates: list[tuple[float, str]]) -> bool:
    """Is the expected number present in the extracted candidates."""
    for value, tag in candidates:
        if kind == "percent":
            if tag not in {"percent", "bare"}:
                continue
            if abs(value - expected) <= PERCENT_TOLERANCE:
                return True
        elif kind == "count":
            if tag not in {"bare", "money"}:
                continue
            if abs(value - expected) <= COUNT_TOLERANCE:
                return True
        else:  # money
            if tag == "percent":
                continue
            tolerance = max(MONEY_ABS_TOLERANCE, abs(expected) * MONEY_REL_TOLERANCE)
            if abs(value - expected) <= tolerance:
                return True
    return False


def _normalise(text: str) -> str:
    return re.sub(r"\s+", " ", (text or "")).lower()


def _any_pattern(text: str, patterns: list[str]) -> bool:
    lowered = _normalise(text)
    return any(re.search(p, lowered) for p in patterns)


def looks_refused(text: str) -> bool:
    return _any_pattern(text, REFUSAL_PATTERNS)


def looks_clarifying(text: str) -> bool:
    return _any_pattern(text, CLARIFY_PATTERNS)


def looks_like_agent_failure(text: str) -> bool:
    """Did the agent itself fail, as opposed to answering badly."""
    return _any_pattern(text, AGENT_FAILURE_PATTERNS)


def looks_time_narrowed(text: str) -> bool:
    """Did the answer narrow the period on a question that set no period."""
    return _any_pattern(text, TIME_NARROWING_PATTERNS)


# --------------------------------------------------------------------------
# Grading
# --------------------------------------------------------------------------

@dataclass
class Attempt:
    question_id: str
    attempt: int
    answer: str
    grade: str
    detail: str = ""
    latency_ms: int = 0
    generated_dax: str = ""


def grade_answer(expected: Expected, answer: str) -> tuple[str, str]:
    """Grade one free-text answer. Returns (grade, human readable detail)."""
    text = answer or ""

    # An agent that fell over has told us nothing about the model.
    if looks_like_agent_failure(text):
        return ERRORED, "the agent failed to produce a result, not a model defect"

    if expected.probe_kind:
        return _grade_probe(expected, text)

    if not text.strip():
        return REFUSED, "empty response"

    candidates = extract_numbers(text)
    lowered = _normalise(text)

    missing_values = [
        f"{value:,.2f} ({kind})"
        for value, kind in expected.values
        if not matches_value(value, kind, candidates)
    ]
    missing_labels = [
        "/".join(group)
        for group in expected.labels
        if not any(alt.lower() in lowered for alt in group)
    ]

    if not missing_values and not missing_labels:
        return CORRECT, "all expected values and labels present"

    # No numbers at all, and the model said it could not answer.
    if not candidates and looks_refused(text):
        return REFUSED, "refused a question it should have answered"

    # Right labels but wrong numbers is a different defect from wrong labels.
    if missing_values and not missing_labels and expected.labels:
        return PARTLY_CORRECT, f"labels right, values missing: {', '.join(missing_values)}"

    if missing_labels and not missing_values:
        return PARTLY_CORRECT, f"values right, labels missing: {', '.join(missing_labels)}"

    detail_parts = []
    if missing_values:
        detail_parts.append(f"values missing: {', '.join(missing_values)}")
    if missing_labels:
        detail_parts.append(f"labels missing: {', '.join(missing_labels)}")
    return WRONG, "; ".join(detail_parts)


def _grade_probe(expected: Expected, text: str) -> tuple[str, str]:
    """Grade F01 to F03, where declining or disclosing is the correct outcome.

    Acceptance is driven by the per-probe rules in PROBE_ACCEPT, which are
    written from the "good outcome" column of the question bank. That matters
    because a well-behaved answer often contains no refusal language at all.
    "Northwest is not a valid region. The valid regions are Central, East and
    West" is the perfect answer and contains no "cannot" anywhere.
    """
    if not text.strip():
        return REFUSED, "empty response, which is not the same as a good refusal"

    lowered = _normalise(text)
    accepted = any(re.search(p, lowered) for p in expected.probe_accept)

    if accepted:
        # Naming the nonexistent thing in order to deny it is correct.
        return CORRECT, {
            "refuse": "declined and explained, which is the good outcome",
            "clarify": "clarified or disclosed its interpretation",
        }.get(expected.probe_kind, "behaved as expected")

    for word in expected.forbidden:
        if word in lowered:
            return WRONG, f"reported data for the nonexistent entity '{word}'"

    if expected.probe_kind == "refuse":
        return WRONG, "answered a question it should have declined"
    if expected.probe_kind == "clarify":
        return WRONG, "picked an interpretation silently"
    return WRONG, "unknown probe kind"


# --------------------------------------------------------------------------
# Classification across repetitions
# --------------------------------------------------------------------------

@dataclass
class QuestionResult:
    question_id: str
    kind: str
    attempts: list[Attempt] = field(default_factory=list)

    @property
    def grades(self) -> list[str]:
        return [a.grade for a in self.attempts]

    @property
    def correct_count(self) -> int:
        return sum(1 for g in self.grades if g == CORRECT)

    @property
    def error_count(self) -> int:
        return sum(1 for g in self.grades if g == ERRORED)

    @property
    def classification(self) -> str:
        return classify_attempts(self.grades)

    @property
    def median_latency_ms(self) -> int:
        values = [a.latency_ms for a in self.attempts if a.latency_ms]
        return int(statistics.median(values)) if values else 0

    @property
    def is_defect(self) -> bool:
        return self.classification in {STABLE_FAILURE, FLAKE, ERRORED_RUN}


def classify_attempts(grades: list[str]) -> str:
    """Stable pass, stable failure, flake, or errored.

    Attempts where the agent itself fell over are excluded before judging the
    model. An infrastructure failure counted as a wrong answer would turn a
    healthy model into a false flake, and a metric people learn to discount is
    worse than no metric at all.

    A flake is the interesting case and it is why the harness repeats every
    question. A single run cannot tell a model that is wrong from a model that
    is ambiguous, and the second is worse in front of an audience because you
    cannot predict it or brief around it.
    """
    if not grades:
        return STABLE_FAILURE

    valid = [g for g in grades if g != ERRORED]
    if not valid:
        return ERRORED_RUN

    correct = sum(1 for g in valid if g == CORRECT)
    if correct == len(valid):
        return STABLE_PASS
    if correct == 0:
        return STABLE_FAILURE
    return FLAKE


def score_run(results: list[QuestionResult]) -> dict:
    """Summarise a run. Only scored questions count toward the /15."""
    scored = [r for r in results if r.kind == SCORED]
    probes = [r for r in results if r.kind == PROBE]

    passed = sum(1 for r in scored if r.classification == STABLE_PASS)
    flakes = [r.question_id for r in results if r.classification == FLAKE]
    failures = [r.question_id for r in results if r.classification == STABLE_FAILURE]
    errored = [r.question_id for r in results if r.classification == ERRORED_RUN]
    guardrails_lost = [
        r.question_id for r in probes if r.classification not in {STABLE_PASS, ERRORED_RUN}
    ]
    latencies = [r.median_latency_ms for r in results if r.median_latency_ms]
    attempt_count = sum(len(r.attempts) for r in results)
    error_attempts = sum(r.error_count for r in results)

    return {
        "score": passed,
        "max_score": len(scored),
        "flake_count": len(flakes),
        "flake_questions": flakes,
        "failure_questions": failures,
        "errored_questions": errored,
        "guardrails_lost": guardrails_lost,
        "median_latency_ms": int(statistics.median(latencies)) if latencies else 0,
        "attempt_count": attempt_count,
        "error_attempts": error_attempts,
        "error_rate": (error_attempts / attempt_count) if attempt_count else 0.0,
    }


# --------------------------------------------------------------------------
# Defect routing
# --------------------------------------------------------------------------

@dataclass(frozen=True)
class FixProposal:
    question_id: str
    classification: str
    tier: int
    fix_target: str
    rationale: str
    automatable: bool


# Tier 1 is additive metadata only, and the bot may open a pull request.
# Tier 2 changes semantics or numbers, so the bot opens an issue and a human
# writes the fix.
# Tier 3 is wording, or a verified answer, and is never automated at all.
TIER_ACTION = {
    0: "no model change, investigate the run itself",
    1: "bot opens a pull request, human merges",
    2: "bot opens an issue with evidence, human writes the fix",
    3: "human only, never automated",
}


def route_defect(result: QuestionResult, expected: Expected) -> FixProposal:
    """Map an observed failure to a fix class and an automation tier.

    This is the guarded part of the loop. It never edits anything. It decides
    what kind of change would plausibly help and who is allowed to make it.
    """
    qid = result.question_id
    classification = result.classification
    grades = set(result.grades)
    detail = " ".join(a.detail for a in result.attempts).lower()
    answers = " ".join(a.answer for a in result.attempts).lower()

    # The agent fell over on every attempt. Nothing has been learned about the
    # model, so proposing a model change would be guessing.
    if classification == ERRORED_RUN:
        return FixProposal(
            qid, classification, 0,
            "no model change",
            "The agent failed to produce a result on every attempt. This is an "
            "infrastructure or capacity problem, not a modelling one. Re-run "
            "before drawing any conclusion.",
            automatable=False,
        )

    # A lost guardrail is the most serious outcome and it is invisible to the
    # score, because F01 to F03 sit outside the /15.
    if expected.probe_kind:
        return FixProposal(
            qid, classification, 1,
            "semantic-model/ai-instructions.md",
            "A guardrail probe stopped behaving. Restore the constraint that "
            "tells the model it holds historical data only, that margin is "
            "ambiguous, or which regions exist.",
            automatable=True,
        )

    # The answer admits it narrowed the period on a question that set no
    # period. That is a missing default, which is additive metadata, and it
    # does not require anyone to change a measure.
    if classification != STABLE_PASS and looks_time_narrowed(answers):
        return FixProposal(
            qid, classification, 1,
            "semantic-model/ai-instructions.md, default time scope",
            "Silently narrowed to the most recent period when the question "
            "carried no time filter. Add an instruction that a question "
            "without a stated period covers all available data.",
            automatable=True,
        )

    if classification == FLAKE:
        return FixProposal(
            qid, classification, 2,
            "semantic-model metadata, ambiguity",
            "Answered correctly on some attempts and not others. That is "
            "ambiguity rather than a wrong definition, and the usual cause is "
            "two plausible columns or measures with nothing to choose between "
            "them. Needs a human to decide which one is right.",
            automatable=False,
        )

    if REFUSED in grades:
        return FixProposal(
            qid, classification, 1,
            "AI data schema, inclusion",
            "Refused a question it should be able to answer. The usual cause "
            "is that the measure or column is not in the AI data schema.",
            automatable=True,
        )

    if PARTLY_CORRECT in grades and "labels right" in detail:
        return FixProposal(
            qid, classification, 2,
            "measure definition or filter context",
            "Grouped on the right thing and returned the wrong numbers. That "
            "is a measure or filter problem, so it changes a number and needs "
            "a human.",
            automatable=False,
        )

    if PARTLY_CORRECT in grades:
        return FixProposal(
            qid, classification, 1,
            "column and measure descriptions",
            "Found the right numbers under the wrong labels, which is usually "
            "a similarly named column chosen without a description to "
            "distinguish it.",
            automatable=True,
        )

    return FixProposal(
        qid, classification, 2,
        "measure selection, likely Gross Sales versus Total Net Sales",
        "Returned a confident wrong number. On this model the usual cause is "
        "the wrong revenue measure. Confirm against the generated DAX before "
        "changing anything.",
        automatable=False,
    )


def propose_fixes(
    results: list[QuestionResult], expectations: dict[str, Expected]
) -> list[FixProposal]:
    """Propose a fix for every defect. Proposals are not changes."""
    proposals = []
    for result in results:
        if not result.is_defect:
            continue
        expected = expectations.get(result.question_id)
        if expected is None:
            continue
        proposals.append(route_defect(result, expected))
    return proposals


# --------------------------------------------------------------------------
# Alert conditions
# --------------------------------------------------------------------------

def alert_conditions(summary: dict, previous_score: int | None) -> list[dict]:
    """Decide what, if anything, should wake somebody up.

    Returned in priority order. The notebook writes these into the Delta table
    that Activator watches, so the thresholds live here in testable code
    rather than being buried in a rule definition in the portal.
    """
    alerts: list[dict] = []

    if summary["guardrails_lost"]:
        alerts.append({
            "severity": "high",
            "condition": "guardrail_lost",
            "detail": (
                "Probes stopped refusing: "
                + ", ".join(summary["guardrails_lost"])
                + ". The model is answering questions it should decline, and "
                "no score threshold catches this because the probes sit "
                "outside the /15."
            ),
        })

    if previous_score is not None and summary["score"] <= previous_score - 2:
        alerts.append({
            "severity": "high",
            "condition": "score_regression",
            "detail": (
                f"Score fell from {previous_score} to {summary['score']}. "
                "Correlate with the most recent semantic model change."
            ),
        })

    if summary["failure_questions"]:
        alerts.append({
            "severity": "high",
            "condition": "stable_failure",
            "detail": "Reproducible failures: " + ", ".join(summary["failure_questions"]),
        })

    if summary["flake_questions"]:
        alerts.append({
            "severity": "high",
            "condition": "flake",
            "detail": (
                "Nondeterministic answers: "
                + ", ".join(summary["flake_questions"])
                + ". Ambiguity, not a wrong definition."
            ),
        })

    if summary["score"] < 13:
        alerts.append({
            "severity": "medium",
            "condition": "below_floor",
            "detail": f"Score {summary['score']} is below the agreed floor of 13.",
        })

    if summary.get("error_rate", 0) > 0.1:
        alerts.append({
            "severity": "medium",
            "condition": "agent_errors",
            "detail": (
                f"{summary['error_attempts']} of {summary['attempt_count']} "
                "attempts failed before producing a result. That is capacity or "
                "service health, not model quality, and it makes this run's "
                "score less trustworthy."
            ),
        })

    return alerts

## 3. Embedded agent client

Generated from `validation/agent_client.py`. Standard library only.
Opens a fresh MCP session per question so no context leaks between
questions.

In [ ]:
import json
import subprocess
import time
import urllib.error
import urllib.request
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass

FABRIC_RESOURCE = "https://api.fabric.microsoft.com"
PROTOCOL_VERSION = "2025-06-18"


def mcp_url(workspace_id: str, data_agent_id: str) -> str:
    return (
        f"https://api.fabric.microsoft.com/v1/mcp/workspaces/{workspace_id}"
        f"/dataagents/{data_agent_id}/agent"
    )


# --------------------------------------------------------------------------
# Tokens
# --------------------------------------------------------------------------

def token_from_notebookutils() -> str:
    import notebookutils  # noqa: PLC0415

    return notebookutils.credentials.getToken(FABRIC_RESOURCE)


def token_from_azure_cli() -> str:
    result = subprocess.run(
        [
            "az", "account", "get-access-token",
            "--resource", FABRIC_RESOURCE,
            "--query", "accessToken", "-o", "tsv",
        ],
        capture_output=True, text=True, shell=True, check=True,
    )
    return result.stdout.strip()


def get_token() -> str:
    """Prefer the notebook identity, fall back to the Azure CLI."""
    try:
        return token_from_notebookutils()
    except Exception:  # noqa: BLE001 - not running inside a notebook
        return token_from_azure_cli()


# --------------------------------------------------------------------------
# Minimal MCP streamable HTTP client
# --------------------------------------------------------------------------

class McpError(RuntimeError):
    pass


def parse_sse(body: str) -> list[dict]:
    """Pull JSON payloads out of a text/event-stream response."""
    messages = []
    for line in body.splitlines():
        line = line.strip()
        if not line.startswith("data:"):
            continue
        payload = line[len("data:"):].strip()
        if not payload or payload == "[DONE]":
            continue
        try:
            messages.append(json.loads(payload))
        except json.JSONDecodeError:
            continue
    return messages


class McpSession:
    """One MCP session against one endpoint.

    Implements only what the data agent needs: initialize, tools/list and
    tools/call.
    """

    def __init__(self, url: str, token: str, timeout: float = 300.0) -> None:
        self.url = url
        self.token = token
        self.timeout = timeout
        self.session_id: str | None = None
        self._next_id = 0

    def _headers(self) -> dict:
        headers = {
            "Authorization": f"Bearer {self.token}",
            "Content-Type": "application/json",
            "Accept": "application/json, text/event-stream",
            "MCP-Protocol-Version": PROTOCOL_VERSION,
        }
        if self.session_id:
            headers["Mcp-Session-Id"] = self.session_id
        return headers

    def _post(self, payload: dict, expect_reply: bool = True) -> dict | None:
        data = json.dumps(payload).encode("utf-8")
        request = urllib.request.Request(
            self.url, data=data, headers=self._headers(), method="POST"
        )
        try:
            with urllib.request.urlopen(request, timeout=self.timeout) as response:
                session_id = response.headers.get("Mcp-Session-Id")
                if session_id:
                    self.session_id = session_id
                body = response.read().decode("utf-8", errors="replace")
                content_type = (response.headers.get("Content-Type") or "").lower()
        except urllib.error.HTTPError as exc:
            detail = exc.read().decode("utf-8", errors="replace")[:600]
            raise McpError(f"HTTP {exc.code} from MCP endpoint: {detail}") from None

        if not expect_reply:
            return None

        if "text/event-stream" in content_type:
            messages = parse_sse(body)
        elif body.strip():
            messages = [json.loads(body)]
        else:
            messages = []

        target = payload.get("id")
        for message in messages:
            if message.get("id") == target:
                if "error" in message:
                    raise McpError(json.dumps(message["error"])[:600])
                return message.get("result", {})

        if messages:
            return messages[-1].get("result", {})
        raise McpError("no JSON-RPC reply from the MCP endpoint")

    def _call(self, method: str, params: dict) -> dict:
        self._next_id += 1
        return self._post(
            {"jsonrpc": "2.0", "id": self._next_id, "method": method, "params": params}
        ) or {}

    def initialize(self) -> None:
        self._call(
            "initialize",
            {
                "protocolVersion": PROTOCOL_VERSION,
                "capabilities": {},
                "clientInfo": {"name": "contoso-coffee-eval", "version": "1.0"},
            },
        )
        self._post(
            {"jsonrpc": "2.0", "method": "notifications/initialized"},
            expect_reply=False,
        )

    def list_tools(self) -> list[dict]:
        return self._call("tools/list", {}).get("tools", [])

    def call_tool(self, name: str, arguments: dict) -> str:
        result = self._call("tools/call", {"name": name, "arguments": arguments})
        blocks = result.get("content", []) or []
        texts = [b.get("text", "") for b in blocks if b.get("type") == "text"]
        return "\n".join(t for t in texts if t)


# --------------------------------------------------------------------------
# Client
# --------------------------------------------------------------------------

@dataclass
class AgentReply:
    question: str
    answer: str
    latency_ms: int
    error: str = ""


class DataAgentClient:
    """Ask a published data agent questions.

    Parameters
    ----------
    concurrency:
        How many questions to have in flight at once. Kept low by default.
        The agent is a shared capacity resource, and throttling produces
        failures that look exactly like a flake, which would poison the one
        signal this harness exists to produce.
    """

    def __init__(
        self,
        workspace_id: str,
        data_agent_id: str,
        token: str | None = None,
        timeout: float = 300.0,
        concurrency: int = 3,
    ) -> None:
        self.url = mcp_url(workspace_id, data_agent_id)
        self.token = token or get_token()
        self.timeout = timeout
        self.concurrency = max(1, concurrency)

    def ask_one(self, question: str) -> AgentReply:
        started = time.monotonic()
        try:
            session = McpSession(self.url, self.token, self.timeout)
            session.initialize()
            tools = session.list_tools()
            if not tools:
                raise McpError("the data agent exposed no tools, is it published")
            tool = tools[0]
            schema = tool.get("inputSchema") or tool.get("input_schema") or {}
            properties = schema.get("properties") or {}
            if not properties:
                raise McpError(f"tool {tool.get('name')} declared no input properties")
            argument = next(iter(properties))
            answer = session.call_tool(tool["name"], {argument: question})
            elapsed = int((time.monotonic() - started) * 1000)
            return AgentReply(question, answer, elapsed)
        except Exception as exc:  # noqa: BLE001
            elapsed = int((time.monotonic() - started) * 1000)
            return AgentReply(question, "", elapsed, error=f"{type(exc).__name__}: {exc}")

    def ask(self, questions: list[str]) -> list[AgentReply]:
        """Ask every question. Reply order matches the input order."""
        if self.concurrency == 1:
            return [self.ask_one(q) for q in questions]
        with ThreadPoolExecutor(max_workers=self.concurrency) as pool:
            return list(pool.map(self.ask_one, questions))

## 4. Question bank

Embedded verbatim from `validation/question-bank.md`, and parsed rather
than retyped. A question asked by the harness that has drifted from the
question printed in the docs is a silent and very confusing failure.

In [ ]:
QUESTION_BANK_MD = r'''
# Question bank

Fifteen questions. Ask them **exactly as written**, in this order, of every AI surface
you test. Do not reword them to get a better answer. A reworded question is a hidden
failure.

Get the correct answers with:

```bash
python validation/ground_truth.py
```

Never write an expected value by hand. The data generator is seeded, so these values are
identical for everyone who runs the demo.

---

| # | Question | What it tests |
| --- | --- | --- |
| Q01 | What is our total net revenue? | Does it pick the right measure at all |
| Q02 | What is our total gross margin? | Derived measure, dollars not percent |
| Q03 | What is our gross margin percentage? | Ratio measure, not a sum of ratios |
| Q04 | How many units did we sell? | Units vs order lines vs customers |
| Q05 | What was net revenue in 2024? | Date table and year filter |
| Q06 | What was net revenue in 2025? | Same, second year |
| Q07 | How much did revenue grow in 2025 compared to 2024? | Time intelligence, percentage growth |
| Q08 | Which store has the highest net revenue? | Ranking across a dimension |
| Q09 | Which product has the highest net revenue? | Ranking across a second dimension |
| Q10 | Show me net revenue by region. | Grouping, and the verified answer path |
| Q11 | Show me net revenue by product category. | Grouping on a second dimension |
| Q12 | Break down net revenue by sales channel. | A column on the fact table |
| Q13 | Which month in 2025 had the highest net revenue? | Filter plus rank plus date grain |
| Q14 | Compare weekend and weekday net revenue. | A boolean flag column |
| Q15 | What is the average value of a sales order line? | Ratio of two measures |

---

## The three questions that should fail

Ask these too, and record **how** they fail. A good failure is more useful in a demo
than a good answer.

| # | Question | The good outcome |
| --- | --- | --- |
| F01 | What will revenue be next quarter? | Says the model contains historical data only, does not project |
| F02 | Which store is most profitable? | Asks whether you mean margin dollars or margin rate, or states that it used margin dollars |
| F03 | Show me sales for the Northwest region. | Says there is no Northwest region, and lists the three that exist |

Before phase 4, all three usually fail badly. F01 invents a projection, F02 silently
picks one interpretation, F03 quietly substitutes West. After the AI instructions in
phase 4, they should behave. That contrast is the demo.

---

## How to score

| Grade | Meaning |
| --- | --- |
| Correct | Right number, right grouping, right filter |
| Partly correct | Right shape, wrong filter or wrong measure |
| Wrong | Wrong number, or invented data |
| Refused | Said it could not answer. Sometimes this is the right answer, see F01 to F03 |

Judge the number, not the wording. AI is nondeterministic and the sentence will change
between runs. That is expected and it is not a failure.

For every answer that is not Correct, expand **How Copilot arrived at this** and record
which fields, measures, and filters were used. That is your diagnosis.

Record everything in [`scorecard.md`](scorecard.md).

'''

questions = parse_question_bank(QUESTION_BANK_MD)
print(f"parsed {len(questions)} questions")
for q in questions:
    print(f"  {q.id} [{q.kind}] {q.text}")


## 5. Ground truth from the lakehouse

Computed with Spark straight off the Delta tables, independently of the
semantic model. That is the point: if the oracle came from the same
semantic model the agent queries, a modelling error would move the
answer and the expected value together and the test would pass while
being wrong.

The mirror of `ground_truth.compute_raw()`, which does the same sums
over the committed CSVs.

In [ ]:
from pyspark.sql import functions as F

lh = f"{LAKEHOUSE_NAME}."

sales = spark.table(lh + "fact_sales")
dim_date = spark.table(lh + "dim_date")
dim_store = spark.table(lh + "dim_store")
dim_product = spark.table(lh + "dim_product")

joined = (
    sales.join(dim_date, "date_key")
         .join(dim_store, "store_key")
         .join(dim_product, "product_key")
)

totals = joined.agg(
    F.sum("net_amount").alias("total_net"),
    F.sum("cost_amount").alias("total_cost"),
    F.sum("quantity").alias("total_units"),
    F.count(F.lit(1)).alias("line_count"),
).collect()[0]

total_net = float(totals["total_net"])
total_margin = total_net - float(totals["total_cost"])


def as_map(df, key_col, order_desc=True):
    rows = (
        df.groupBy(key_col)
          .agg(F.sum("net_amount").alias("net"))
          .orderBy(F.col("net").desc() if order_desc else F.col("net"))
          .collect()
    )
    return {str(r[key_col]): float(r["net"]) for r in rows}


by_year = as_map(joined, "year")
by_region = as_map(joined, "region")
by_store = as_map(joined, "store_name")
by_category = as_map(joined, "category")
by_product = as_map(joined, "product_name")
by_channel = as_map(joined, "channel")
by_month_2025 = as_map(joined.filter(F.col("year") == 2025), "year_month")

weekend_rows = (
    joined.groupBy("is_weekend").agg(F.sum("net_amount").alias("net")).collect()
)
weekend_net = next(
    (float(r["net"]) for r in weekend_rows if str(r["is_weekend"]).lower() in ("true", "1")),
    0.0,
)
weekday_net = next(
    (float(r["net"]) for r in weekend_rows if str(r["is_weekend"]).lower() not in ("true", "1")),
    0.0,
)

net_2024 = by_year.get("2024", 0.0)
net_2025 = by_year.get("2025", 0.0)


def top(mapping):
    key = max(mapping, key=mapping.get)
    return (key, mapping[key])


raw = {
    "total_net": total_net,
    "total_margin": total_margin,
    "margin_pct": total_margin / total_net,
    "total_units": int(totals["total_units"]),
    "net_2024": net_2024,
    "net_2025": net_2025,
    "yoy_pct": (net_2025 - net_2024) / net_2024,
    "top_store": top(by_store),
    "top_product": top(by_product),
    "by_region": by_region,
    "by_category": by_category,
    "by_channel": by_channel,
    "best_month_2025": top(by_month_2025),
    "weekend_net": weekend_net,
    "weekday_net": weekday_net,
    "avg_order_line": total_net / int(totals["line_count"]),
}

expectations = build_expectations(raw)

print(f"total net revenue : ${raw['total_net']:,.2f}")
print(f"gross margin      : ${raw['total_margin']:,.2f} ({raw['margin_pct']:.2%})")
print(f"units             : {raw['total_units']:,}")
print(f"top store         : {raw['top_store'][0]} (${raw['top_store'][1]:,.2f})")
print(f"regions           : {list(raw['by_region'])}")
print(f"\nbuilt {len(expectations)} expectations")


## 6. Run the evaluation

Every question, `REPEAT` times, each in a fresh session.

In [ ]:
import uuid
from datetime import datetime, timezone

run_id = str(uuid.uuid4())
run_ts = datetime.now(timezone.utc)

client = DataAgentClient(WORKSPACE_ID, DATA_AGENT_ID, concurrency=CONCURRENCY)
results = {q.id: QuestionResult(q.id, q.kind) for q in questions}

for attempt in range(1, REPEAT + 1):
    print(f"--- attempt {attempt} of {REPEAT} ---", flush=True)
    replies = client.ask([q.text for q in questions])

    for question, reply in zip(questions, replies):
        if reply.error:
            grade, detail = ERRORED, f"transport error: {reply.error[:200]}"
        else:
            grade, detail = grade_answer(expectations[question.id], reply.answer)

        results[question.id].attempts.append(
            Attempt(
                question_id=question.id,
                attempt=attempt,
                answer=reply.answer,
                grade=grade,
                detail=detail,
                latency_ms=reply.latency_ms,
            )
        )
        flag = "ok  " if grade == CORRECT else "FAIL"
        print(f"  {flag} {question.id} {grade:<15} {detail[:80]}", flush=True)

ordered = [results[q.id] for q in questions]
print(f"\nrun_id {run_id}")


## 7. Score, classify and propose

Proposals are proposals. Nothing here edits the model.

In [ ]:
summary = score_run(ordered)
proposals = propose_fixes(ordered, expectations)

print(f"score             : {summary['score']} / {summary['max_score']}")
print(f"flakes            : {summary['flake_questions'] or 'none'}")
print(f"stable failures   : {summary['failure_questions'] or 'none'}")
print(f"errored questions : {summary['errored_questions'] or 'none'}")
print(f"guardrails lost   : {summary['guardrails_lost'] or 'none'}")
print(f"agent errors      : {summary['error_attempts']} / {summary['attempt_count']}")
print(f"median latency ms : {summary['median_latency_ms']}")

if proposals:
    print("\nproposed fixes, none of which are applied automatically:")
    for p in proposals:
        print(f"  {p.question_id}  tier {p.tier}  {TIER_ACTION[p.tier]}")
        print(f"      target   : {p.fix_target}")
        print(f"      rationale: {p.rationale}")
else:
    print("\nno defects, so nothing to propose")


## 8. Write the Delta tables

Three append only tables. History is the whole point: correlating a
score drop with a model change is what turns an alert into a diagnosis.

| Table | Grain |
| --- | --- |
| `eval_runs` | one row per run, and the row Activator watches |
| `eval_results` | one row per question per attempt |
| `eval_defects` | one open defect per failing question, with its proposal |


In [ ]:
from pyspark.sql import Row
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, TimestampType, BooleanType,
)

runs_table = f"{TABLE_PREFIX}_runs"
results_table = f"{TABLE_PREFIX}_results"
defects_table = f"{TABLE_PREFIX}_defects"

# Previous score, so a regression can be detected rather than just a low score.
previous_score = None
if spark.catalog.tableExists(lh + runs_table):
    prior = (
        spark.table(lh + runs_table)
             .filter(F.col("surface") == SURFACE)
             .orderBy(F.col("run_ts").desc())
             .limit(1)
             .collect()
    )
    if prior:
        previous_score = int(prior[0]["score"])

alerts = alert_conditions(summary, previous_score)
alert_severity = "high" if any(a["severity"] == "high" for a in alerts) else (
    "medium" if alerts else "none"
)

runs_schema = StructType([
    StructField("run_id", StringType()),
    StructField("run_ts", TimestampType()),
    StructField("surface", StringType()),
    StructField("workspace_id", StringType()),
    StructField("data_agent_id", StringType()),
    StructField("repeat_count", IntegerType()),
    StructField("score", IntegerType()),
    StructField("max_score", IntegerType()),
    StructField("previous_score", IntegerType()),
    StructField("flake_count", IntegerType()),
    StructField("failure_count", IntegerType()),
    StructField("guardrails_lost_count", IntegerType()),
    StructField("errored_count", IntegerType()),
    StructField("error_attempts", IntegerType()),
    StructField("attempt_count", IntegerType()),
    StructField("median_latency_ms", IntegerType()),
    StructField("alert_count", IntegerType()),
    StructField("alert_severity", StringType()),
    StructField("alert_detail", StringType()),
])

runs_row = Row(
    run_id=run_id,
    run_ts=run_ts,
    surface=SURFACE,
    workspace_id=WORKSPACE_ID,
    data_agent_id=DATA_AGENT_ID,
    repeat_count=int(REPEAT),
    score=int(summary["score"]),
    max_score=int(summary["max_score"]),
    previous_score=previous_score,
    flake_count=int(summary["flake_count"]),
    failure_count=len(summary["failure_questions"]),
    guardrails_lost_count=len(summary["guardrails_lost"]),
    errored_count=len(summary["errored_questions"]),
    error_attempts=int(summary["error_attempts"]),
    attempt_count=int(summary["attempt_count"]),
    median_latency_ms=int(summary["median_latency_ms"]),
    alert_count=len(alerts),
    alert_severity=alert_severity,
    alert_detail=" | ".join(f"[{a['severity']}] {a['condition']}: {a['detail']}" for a in alerts),
)

runs_df = spark.createDataFrame([runs_row], schema=runs_schema)
runs_df.write.mode("append").format("delta").saveAsTable(lh + runs_table)

results_schema = StructType([
    StructField("run_id", StringType()),
    StructField("run_ts", TimestampType()),
    StructField("question_id", StringType()),
    StructField("kind", StringType()),
    StructField("attempt", IntegerType()),
    StructField("grade", StringType()),
    StructField("detail", StringType()),
    StructField("classification", StringType()),
    StructField("latency_ms", IntegerType()),
    StructField("answer", StringType()),
])

result_rows = [
    Row(
        run_id=run_id, run_ts=run_ts, question_id=r.question_id, kind=r.kind,
        attempt=int(a.attempt), grade=a.grade, detail=a.detail,
        classification=r.classification, latency_ms=int(a.latency_ms), answer=a.answer,
    )
    for r in ordered for a in r.attempts
]

spark.createDataFrame(result_rows, schema=results_schema) \
     .write.mode("append").format("delta").saveAsTable(lh + results_table)

defects_schema = StructType([
    StructField("run_id", StringType()),
    StructField("run_ts", TimestampType()),
    StructField("question_id", StringType()),
    StructField("classification", StringType()),
    StructField("tier", IntegerType()),
    StructField("fix_target", StringType()),
    StructField("rationale", StringType()),
    StructField("automatable", BooleanType()),
    StructField("action", StringType()),
    StructField("status", StringType()),
])

if proposals:
    defect_rows = [
        Row(
            run_id=run_id, run_ts=run_ts, question_id=p.question_id,
            classification=p.classification, tier=int(p.tier),
            fix_target=p.fix_target, rationale=p.rationale,
            automatable=bool(p.automatable), action=TIER_ACTION[p.tier],
            status="awaiting_human_confirmation",
        )
        for p in proposals
    ]
    spark.createDataFrame(defect_rows, schema=defects_schema) \
         .write.mode("append").format("delta").saveAsTable(lh + defects_table)
else:
    # Create the table even on a clean run so Activator has something to bind.
    spark.createDataFrame([], schema=defects_schema) \
         .write.mode("append").format("delta").saveAsTable(lh + defects_table)

print(f"wrote {runs_table}, {results_table}, {defects_table}")
print(f"previous score {previous_score} -> {summary['score']}")


## 9. Publish to the Eventhouse for Activator

Activator cannot watch a Delta table. It watches a KQL query, so the
run summary is published to the Eventhouse as well. Delta stays the
system of record and the thing you query; Kusto is the event spine
that makes alerting possible.

One row per run, carrying the alert verdict the tested Python already
reached. The rule in Activator only has to read `alert_severity`,
which keeps the thresholds in code that has unit tests rather than in
a rule definition nobody can test.

In [ ]:
import notebookutils

kusto_token = notebookutils.credentials.getToken(KUSTO_URI)

(
    runs_df.write.format("com.microsoft.kusto.spark.synapse.datasource")
    .option("kustoCluster", KUSTO_URI)
    .option("kustoDatabase", KUSTO_DB)
    .option("kustoTable", "eval_runs")
    .option("accessToken", kusto_token)
    .option("tableCreateOptions", "CreateIfNotExist")
    .mode("Append")
    .save()
)

print(f"published run {run_id} to {KUSTO_DB}.eval_runs")
print(f"alert_severity={alert_severity} alert_count={len(alerts)}")


## 10. Alert payload

The thresholds live in tested Python rather than being buried in a rule
definition in the portal. Activator reads `alert_severity` and
`alert_count` off `eval_runs` and decides whether to notify.

In [ ]:
if alerts:
    print(f"{len(alerts)} alert(s), highest severity {alert_severity}\n")
    for a in alerts:
        print(f"[{a['severity']}] {a['condition']}")
        print(f"    {a['detail']}\n")
    print(
        "Activator watches eval_runs. A human confirms the defect before any "
        "fix is written, and no proposal may ever add a verified answer."
    )
else:
    print("no alerts, nothing to confirm")

display(
    spark.table(lh + runs_table)
         .orderBy(F.col("run_ts").desc())
         .select(
             "run_ts", "surface", "score", "previous_score", "flake_count",
             "failure_count", "guardrails_lost_count", "alert_severity",
         )
         .limit(10)
)
